In [2]:
import ee

In [4]:
ee.Authenticate()
ee.Initialize(project='cameroon-infra-geospatial')

Enter verification code:  4/1AdkVLPx4Tgr5WNumw1eegiPc89NmT2cynWvtaVS7lWrHauSNwlweLTwI6HA



Successfully saved authorization token.


In [5]:
import geopandas as gpd
import json

# 노트북이 레포 루트에 있으므로 상대경로는 data/processed/ (../ 없이)
gdf = gpd.read_file("../data/processed/douala_yaounde_districts.geojson")

geojson_dict = json.loads(gdf.to_json())
districts_fc = ee.FeatureCollection(geojson_dict)

print(f"구역 수: {districts_fc.size().getInfo()}")

구역 수: 13


In [9]:
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterDate('2025-01-01', '2026-01-01') \
    .select('avg_rad') \
    .mean()

def get_mean_radiance(feature):
    mean_dict = viirs.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=feature.geometry(),
        scale=500
    )
    return feature.set('mean_radiance', mean_dict.get('avg_rad'))

districts_with_viirs = districts_fc.map(get_mean_radiance)
results = districts_with_viirs.getInfo()
for f in results['features']:
    print(f['properties']['district_name'], '-', f['properties']['mean_radiance'])

Douala 1er - 13.997539570551762
Douala 2e - 10.254958609058798
Douala 3e - 6.136907941536896
Douala 4e - 2.2876669463760844
Douala 5e - 4.262470436230381
Douala 6e - 0.3676910599388305
Yaounde 1er - 14.165397811868724
Yaounde 2e - 10.90152962080052
Yaounde 3e - 7.979102984601804
Yaounde 4e - 13.229416808331605
Yaounde 5e - 18.220760501081127
Yaounde 6e - 13.250827258007911
Yaounde 7e - 6.528635143193935


In [10]:
import pandas as pd

data = []
for f in results['features']:
    data.append({
        'district_name': f['properties']['district_name'],
        'city': f['properties']['city'],
        'mean_radiance': f['properties']['mean_radiance']
    })

df_viirs = pd.DataFrame(data)
df_viirs.to_csv('../data/processed/viirs_nightlight_2025.csv', index=False)
print(df_viirs)

   district_name     city  mean_radiance
0     Douala 1er   Douala      13.997540
1      Douala 2e   Douala      10.254959
2      Douala 3e   Douala       6.136908
3      Douala 4e   Douala       2.287667
4      Douala 5e   Douala       4.262470
5      Douala 6e   Douala       0.367691
6    Yaounde 1er  Yaounde      14.165398
7     Yaounde 2e  Yaounde      10.901530
8     Yaounde 3e  Yaounde       7.979103
9     Yaounde 4e  Yaounde      13.229417
10    Yaounde 5e  Yaounde      18.220761
11    Yaounde 6e  Yaounde      13.250827
12    Yaounde 7e  Yaounde       6.528635
